In [0]:
import pyspark.sql.functions as F

# 1. Carregando as tabelas da Silver com Alias
df_total = spark.table("visagio_rocket_lab.silver.fat_pedido_total").alias("ped")
df_rev = spark.table("visagio_rocket_lab.silver.fat_avaliacoes_pedidos").alias("rev")
df_prod = spark.table("visagio_rocket_lab.silver.dim_produtos").alias("prod")
df_vend = spark.table("visagio_rocket_lab.silver.dim_vendedores").alias("vend")

# 2. Join 

col_id_vend_dim = [c for c in df_vend.columns if "seller" in c or "vendedor" in c][0]

df_analise = df_total.join(df_rev, "id_pedido", "inner") \
    .join(df_prod, "id_produto", "inner") \
    .join(df_vend, F.col("ped.seller_id") == F.col(f"vend.{col_id_vend_dim}"), "inner")

# PROJETO 1: VISÃO COMERCIAL 
df_comercial = df_analise.withColumn("ano_venda", F.year("data_pedido")) \
    .withColumn("mes_venda", F.month("data_pedido")) \
    .groupBy("ano_venda", "mes_venda", "categoria_produto") \
    .agg(
        F.countDistinct("id_pedido").alias("total_pedidos"),
        F.sum("valor_total_pago_brl").alias("receita_total_brl")
    )

# Salvando com overwriteSchema para evitar Mismatch 
df_comercial.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("visagio_rocket_lab.gold.fat_vendas_comercial")

#PROJETO 2: SATISFAÇÃO
df_satisfacao = df_analise.groupBy("categoria_produto", "cidade") \
    .agg(
        F.count("id_avaliacao").alias("total_avaliacoes"),
        F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media")
    )

df_satisfacao.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("visagio_rocket_lab.gold.fat_avaliacoes_clientes")

# Rankings de Qualidade
print("--- PRODUTO MAIS BEM AVALIADO ---")
display(df_analise.groupBy("id_produto").agg(F.avg("nota_avaliacao").alias("media"), F.count("id_avaliacao").alias("vol")).orderBy(F.desc("media"), F.desc("vol")).limit(1))

print("Gold finalizada!")

--- PRODUTO MAIS BEM AVALIADO ---


id_produto,media,vol
ebf9bc6cd600eadd681384e3116fda85,5.0,44


Gold finalizada!
